# MCI-GRU Full Feature Factorial Ablation - Google Colab

This notebook follows `notebooks/ablation_evaluation_loop_colab.ipynb`, but it is built for the full decision run: 20 ensemble members, 100 epochs, early stopping patience 15, and 1,000 bootstrap resamples for every primary ablation.

Primary matrix: momentum speed selection x graph construction x regime feature set.

## 1. Mount Drive, Clone Repo, Install Dependencies

In [ ]:
from google.colab import drive
from pathlib import Path
import os
import subprocess
import sys

drive.mount('/content/drive')

REPO_URL = 'https://github.com/magilliam27/MCI-GRU.git'
BRANCH = 'main'
REPO_DIR = Path('/content/MCI-GRU')
DRIVE_ROOT = Path('/content/drive/MyDrive/MCI-GRU-Ablations')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

if not REPO_DIR.exists():
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}
else:
    os.chdir(REPO_DIR)
    !git fetch origin
    !git checkout {BRANCH}
    !git pull origin {BRANCH}

os.chdir(REPO_DIR)
print('Working directory:', Path.cwd())

!python -m pip install -q --upgrade pip setuptools wheel
!python -m pip install -q -r requirements.txt
!python -m pip install -q -e ".[dev,tracking,fred]"

REQUIRE_GPU = True

try:
    import torch
except ImportError as exc:
    raise RuntimeError('PyTorch is not installed; rerun this setup cell.') from exc

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    device_idx = torch.cuda.current_device()
    props = torch.cuda.get_device_properties(device_idx)
    print('CUDA version:', torch.version.cuda)
    print('GPU:', torch.cuda.get_device_name(device_idx))
    print(f'GPU memory: {props.total_memory / (1024 ** 3):.1f} GiB')
else:
    msg = (
        'No CUDA GPU is visible in the notebook kernel. In Colab, choose '
        'Runtime -> Change runtime type -> GPU, then restart and run from the top.'
    )
    if REQUIRE_GPU:
        raise RuntimeError(msg)
    print('WARNING:', msg)

child_probe = subprocess.run(
    [
        sys.executable,
        '-c',
        "import torch; "
        "print('Child Torch:', torch.__version__); "
        "print('Child CUDA available:', torch.cuda.is_available()); "
        "print('Child CUDA version:', torch.version.cuda); "
        "print('Child GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO_GPU')",
    ],
    text=True,
    capture_output=True,
)
print('Child Python:', sys.executable)
print(child_probe.stdout.strip())
if child_probe.returncode != 0:
    print(child_probe.stderr)
    raise RuntimeError('Child Python CUDA probe failed; rerun setup after fixing PyTorch.')
child_cuda_visible = 'Child CUDA available: True' in child_probe.stdout
if REQUIRE_GPU and not child_cuda_visible:
    msg = (
        'The training subprocess cannot see CUDA, so run_experiment.py would train on CPU. '
        'Restart the Colab GPU runtime and rerun setup; if this persists, reinstall a CUDA-enabled torch wheel.'
    )
    raise RuntimeError(msg)

print(f'Repo: {REPO_DIR}')
print(f'Drive outputs: {DRIVE_ROOT}')

## 2. Full Run Configuration

In [ ]:
from datetime import datetime

# The notebook copies the first existing candidate into the repo root and passes it to Hydra.
# Keep the 2019-universe file first if you want continuity with prior ablation notebooks.
DATA_FILE_CANDIDATES = [
    'sp500_2019_universe_data.csv',
    'sp500_data.csv',
]
GDRIVE_DATA_DIR = Path('/content/drive/MyDrive/MCI-GRU-Data')

TRAIN_START = '2019-01-01'
TRAIN_END = '2023-12-31'
VAL_START = '2024-01-08'
VAL_END = '2024-12-31'
TEST_START = '2025-01-08'
TEST_END = '2025-12-31'

# Full-run budget requested for every primary ablation.
NUM_MODELS = 20
NUM_EPOCHS = 100
EARLY_STOPPING_PATIENCE = 15
BOOTSTRAP_RESAMPLES = 1000

BATCH_SIZE = 32
LEARNING_RATE = '5e-5'
SEED = 42

# Regime runs should fail loudly if auxiliary data cannot be loaded; otherwise an "on" run can become zeros.
REGIME_STRICT_FOR_REGIME_RUNS = True
REGIME_INPUTS_CSV = ''  # Optional path, e.g. data/raw/market/regime_inputs.csv or a file copied into the repo.
REGIME_ENFORCE_LAG_DAYS = 0
FRED_API_KEY = ''
os.environ['FRED_API_KEY'] = FRED_API_KEY

INCLUDE_DIAGNOSTIC_CONTROLS = True
SKIP_COMPLETED_RUNS = True

RUN_TAG = datetime.now().strftime('%Y%m%d_%H%M%S')
ABLATION_ROOT = DRIVE_ROOT / 'full_feature_factorial' / RUN_TAG
ABLATION_ROOT.mkdir(parents=True, exist_ok=True)

print('Ablation root:', ABLATION_ROOT)
print('Models:', NUM_MODELS)
print('Epochs:', NUM_EPOCHS)
print('Early stopping patience:', EARLY_STOPPING_PATIENCE)
print('Bootstrap resamples:', BOOTSTRAP_RESAMPLES)

## 3. Data Availability Check

In [ ]:
import pandas as pd
import shutil

DATA_FILE = None
for candidate in DATA_FILE_CANDIDATES:
    repo_data = REPO_DIR / candidate
    drive_data = GDRIVE_DATA_DIR / candidate
    if not repo_data.exists() and drive_data.exists():
        shutil.copy2(drive_data, repo_data)
    if repo_data.exists():
        DATA_FILE = candidate
        break

if DATA_FILE is None:
    searched = [str(REPO_DIR / c) for c in DATA_FILE_CANDIDATES] + [str(GDRIVE_DATA_DIR / c) for c in DATA_FILE_CANDIDATES]
    raise FileNotFoundError('Missing market data. Looked for: ' + ', '.join(searched))

repo_data = REPO_DIR / DATA_FILE
preview = pd.read_csv(repo_data, usecols=['dt', 'kdcode'])
preview['dt'] = pd.to_datetime(preview['dt'])
print('Using data file:', repo_data)
print(f'Rows: {len(preview):,}')
print(f'Stocks: {preview.kdcode.nunique():,}')
print(f'Dates: {preview.dt.min().date()} to {preview.dt.max().date()}')
del preview

if REGIME_INPUTS_CSV:
    regime_src = Path(REGIME_INPUTS_CSV)
    regime_repo_path = REPO_DIR / REGIME_INPUTS_CSV
    regime_drive_path = GDRIVE_DATA_DIR / regime_src.name
    if not regime_repo_path.exists() and regime_drive_path.exists():
        regime_repo_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(regime_drive_path, regime_repo_path)
    if not regime_repo_path.exists():
        raise FileNotFoundError(f'REGIME_INPUTS_CSV was set but not found: {regime_repo_path}')
    regime_preview = pd.read_csv(regime_repo_path, nrows=5)
    print('Using regime inputs CSV:', regime_repo_path)
    print('Regime columns:', list(regime_preview.columns))
elif REGIME_STRICT_FOR_REGIME_RUNS and not os.environ.get('FRED_API_KEY'):
    print('WARNING: Regime runs are strict, but FRED_API_KEY is empty and REGIME_INPUTS_CSV is unset.')
    print('Set one of those before running the matrix if you want regime runs to complete.')

## 4. Define Full Factorial Ablation Matrix

In [ ]:
import itertools
import json

MOMENTUM_FACTORS = [
    {
        'factor': 'momentum_static',
        'description': 'Binary momentum with static 50/50 fast/slow blend.',
        'overrides': [
            'features.include_momentum=true',
            'features.include_weekly_momentum=true',
            'features.momentum_encoding=binary',
            'features.momentum_blend_mode=static',
            'features.momentum_blend_fast_weight=0.5',
        ],
    },
    {
        'factor': 'momentum_dynamic',
        'description': 'Binary momentum with cycle-aware dynamic fast/slow blend.',
        'overrides': [
            'features.include_momentum=true',
            'features.include_weekly_momentum=true',
            'features.momentum_encoding=binary',
            'features.momentum_blend_mode=dynamic',
            'features.momentum_dynamic_correction_fast_weight=0.15',
            'features.momentum_dynamic_rebound_fast_weight=0.7',
            'features.momentum_dynamic_lookback_periods=0',
            'features.momentum_dynamic_min_history=252',
            'features.momentum_dynamic_min_state_observations=3',
        ],
    },
]

GRAPH_FACTORS = [
    {
        'factor': 'graph_static_threshold',
        'description': 'Static threshold graph, matching the new default.',
        'overrides': [
            'graph.update_frequency_months=0',
            'graph.top_k=0',
            'graph.top_k_metric=corr',
            'graph.use_multi_feature_edges=true',
            'graph.append_snapshot_age_days=false',
            'graph.use_lead_lag_features=false',
        ],
    },
    {
        'factor': 'graph_dynamic_threshold_6mo',
        'description': 'Dynamic threshold graph rebuilt every 6 months.',
        'overrides': [
            'graph.update_frequency_months=6',
            'graph.top_k=0',
            'graph.top_k_metric=corr',
            'graph.use_multi_feature_edges=true',
            'graph.append_snapshot_age_days=false',
            'graph.use_lead_lag_features=false',
        ],
    },
    {
        'factor': 'graph_dynamic_topk20_pos_6mo',
        'description': 'Dynamic top-K graph with positive-correlation neighbour ranking.',
        'overrides': [
            'graph.update_frequency_months=6',
            'graph.top_k=20',
            'graph.top_k_metric=corr',
            'graph.use_multi_feature_edges=true',
            'graph.append_snapshot_age_days=false',
            'graph.use_lead_lag_features=false',
        ],
    },
    {
        'factor': 'graph_dynamic_topk20_signed_6mo',
        'description': 'Dynamic top-K graph ranked by absolute correlation, preserving signed edge features.',
        'overrides': [
            'graph.update_frequency_months=6',
            'graph.top_k=20',
            'graph.top_k_metric=abs_corr',
            'graph.use_multi_feature_edges=true',
            'graph.append_snapshot_age_days=false',
            'graph.use_lead_lag_features=false',
        ],
    },
]

def regime_overrides(enabled: bool, include_subsequent_returns: bool) -> list[str]:
    overrides = [
        f'features.include_global_regime={str(enabled).lower()}',
        f'features.regime_strict={str(REGIME_STRICT_FOR_REGIME_RUNS and enabled).lower()}',
    ]
    if enabled:
        overrides.extend([
            f'features.regime_include_subsequent_returns={str(include_subsequent_returns).lower()}',
            'features.regime_subsequent_return_horizons=[1,3]',
            'features.regime_change_months=12',
            'features.regime_norm_months=120',
            'features.regime_exclusion_months=1',
            'features.regime_similarity_quantile=0.2',
            'features.regime_min_history_months=24',
            f'features.regime_enforce_lag_days={REGIME_ENFORCE_LAG_DAYS}',
        ])
        if REGIME_INPUTS_CSV:
            overrides.append(f'features.regime_inputs_csv={REGIME_INPUTS_CSV}')
    return overrides

REGIME_FACTORS = [
    {
        'factor': 'regime_off',
        'description': 'No global regime variables.',
        'overrides': regime_overrides(False, False),
    },
    {
        'factor': 'regime_current_only',
        'description': 'Global regime similarity variables without historical subsequent-return features.',
        'overrides': regime_overrides(True, False),
    },
    {
        'factor': 'regime_with_forward_context',
        'description': 'Global regime similarity variables plus similar-history subsequent-return context.',
        'overrides': regime_overrides(True, True),
    },
]

def clean_name(*parts):
    return '__'.join(part.replace('_', '-') for part in parts)

ABLATIONS = []
for momentum, graph, regime in itertools.product(MOMENTUM_FACTORS, GRAPH_FACTORS, REGIME_FACTORS):
    ABLATIONS.append({
        'name': clean_name(momentum['factor'], graph['factor'], regime['factor']),
        'description': ' | '.join([momentum['description'], graph['description'], regime['description']]),
        'family': 'factorial',
        'momentum_factor': momentum['factor'],
        'graph_factor': graph['factor'],
        'regime_factor': regime['factor'],
        'overrides': [*momentum['overrides'], *graph['overrides'], *regime['overrides']],
    })

DIAGNOSTIC_CONTROLS = [
    {
        'name': 'control__base-only__graph-static-threshold__regime-off',
        'description': 'Base OHLCV/turnover only; measures the total value of momentum features.',
        'family': 'diagnostic',
        'momentum_factor': 'momentum_off',
        'graph_factor': 'graph_static_threshold',
        'regime_factor': 'regime_off',
        'overrides': [
            'features.include_momentum=false',
            'features.include_global_regime=false',
            'graph.update_frequency_months=0',
            'graph.top_k=0',
            'graph.use_multi_feature_edges=true',
        ],
    },
    {
        'name': 'control__momentum-static-no-weekly__graph-static-threshold__regime-off',
        'description': 'Static binary momentum without weekly momentum; isolates weekly feature value.',
        'family': 'diagnostic',
        'momentum_factor': 'momentum_static_no_weekly',
        'graph_factor': 'graph_static_threshold',
        'regime_factor': 'regime_off',
        'overrides': [
            'features.include_momentum=true',
            'features.include_weekly_momentum=false',
            'features.momentum_encoding=binary',
            'features.momentum_blend_mode=static',
            'features.include_global_regime=false',
            'graph.update_frequency_months=0',
            'graph.top_k=0',
            'graph.use_multi_feature_edges=true',
        ],
    },
    {
        'name': 'control__momentum-dynamic-no-weekly__graph-static-threshold__regime-off',
        'description': 'Dynamic binary momentum without weekly momentum; checks whether dynamic blend depends on weekly terms.',
        'family': 'diagnostic',
        'momentum_factor': 'momentum_dynamic_no_weekly',
        'graph_factor': 'graph_static_threshold',
        'regime_factor': 'regime_off',
        'overrides': [
            'features.include_momentum=true',
            'features.include_weekly_momentum=false',
            'features.momentum_encoding=binary',
            'features.momentum_blend_mode=dynamic',
            'features.include_global_regime=false',
            'graph.update_frequency_months=0',
            'graph.top_k=0',
            'graph.use_multi_feature_edges=true',
        ],
    },
    {
        'name': 'control__momentum-static__graph-static-scalar-edges__regime-off',
        'description': 'Static graph with legacy scalar edge weights; isolates multi-feature edge attributes.',
        'family': 'diagnostic',
        'momentum_factor': 'momentum_static',
        'graph_factor': 'graph_static_scalar_edges',
        'regime_factor': 'regime_off',
        'overrides': [
            'features.include_momentum=true',
            'features.include_weekly_momentum=true',
            'features.momentum_encoding=binary',
            'features.momentum_blend_mode=static',
            'features.include_global_regime=false',
            'graph.update_frequency_months=0',
            'graph.top_k=0',
            'graph.use_multi_feature_edges=false',
        ],
    },
]

if INCLUDE_DIAGNOSTIC_CONTROLS:
    ABLATIONS.extend(DIAGNOSTIC_CONTROLS)

print('Primary factorial runs:', sum(a['family'] == 'factorial' for a in ABLATIONS))
print('Diagnostic controls:', sum(a['family'] == 'diagnostic' for a in ABLATIONS))
print('Total runs:', len(ABLATIONS))
pd.DataFrame(ABLATIONS)[['name', 'family', 'momentum_factor', 'graph_factor', 'regime_factor', 'description']]

## 5. Helpers: Run, Collect, Score

In [ ]:
import numpy as np
from pathlib import Path

BASE_OVERRIDES = [
    'features=with_momentum',
    'data.source=csv',
    f'data.filename={DATA_FILE}',
    f'data.train_start={TRAIN_START}',
    f'data.train_end={TRAIN_END}',
    f'data.val_start={VAL_START}',
    f'data.val_end={VAL_END}',
    f'data.test_start={TEST_START}',
    f'data.test_end={TEST_END}',
    f'training.num_models={NUM_MODELS}',
    f'training.num_epochs={NUM_EPOCHS}',
    f'training.early_stopping_patience={EARLY_STOPPING_PATIENCE}',
    f'training.batch_size={BATCH_SIZE}',
    f'training.learning_rate={LEARNING_RATE}',
    f'evaluation.bootstrap_resamples={BOOTSTRAP_RESAMPLES}',
    'tracking.enabled=true',
    'tracking.log_predictions=false',
    f'seed={SEED}',
]

def run_ablation(ablation):
    run_dir = ABLATION_ROOT / ablation['name']
    run_dir.mkdir(parents=True, exist_ok=True)
    done_marker = run_dir / 'evaluation_summary.json'
    if SKIP_COMPLETED_RUNS and done_marker.exists():
        print('\n' + '=' * 100)
        print('Skipping completed run:', ablation['name'])
        print('Run dir:', run_dir)
        print('=' * 100)
        return {'name': ablation['name'], 'run_dir': str(run_dir), 'returncode': 0, 'skipped': True}

    cmd = [
        sys.executable,
        '-u',
        'run_experiment.py',
        *BASE_OVERRIDES,
        *ablation['overrides'],
        f'experiment_name={ablation["name"]}',
        f'hydra.run.dir={run_dir}',
    ]
    print('\n' + '=' * 100)
    print('Running:', ablation['name'])
    print(ablation['description'])
    print('Command:', ' '.join(cmd))
    print('=' * 100)
    result = subprocess.run(cmd, cwd=REPO_DIR, text=True, capture_output=True)
    (run_dir / 'stdout.log').write_text(result.stdout)
    (run_dir / 'stderr.log').write_text(result.stderr)
    print(result.stdout[-4000:])
    if result.returncode != 0:
        print(result.stderr[-4000:])
    return {'name': ablation['name'], 'run_dir': str(run_dir), 'returncode': result.returncode, 'skipped': False}

def flatten_json(prefix, obj, out):
    if isinstance(obj, dict):
        for key, value in obj.items():
            flatten_json(f'{prefix}.{key}' if prefix else key, value, out)
    elif isinstance(obj, list):
        out[prefix] = json.dumps(obj)
    else:
        out[prefix] = obj

def load_json(path):
    path = Path(path)
    if path.exists():
        return json.loads(path.read_text())
    return None

def collect_ablation(ablation, status):
    run_dir = Path(status['run_dir'])
    row = {
        'name': ablation['name'],
        'description': ablation['description'],
        'family': ablation.get('family'),
        'momentum_factor': ablation.get('momentum_factor'),
        'graph_factor': ablation.get('graph_factor'),
        'regime_factor': ablation.get('regime_factor'),
        'returncode': status['returncode'],
        'skipped': status.get('skipped', False),
        'run_dir': str(run_dir),
        'overrides': ' '.join(ablation['overrides']),
    }
    for filename, prefix in [
        ('training_summary.json', 'training'),
        ('evaluation_summary.json', 'evaluation'),
        ('run_metadata.json', 'metadata'),
        ('walkforward_summary.json', 'walkforward'),
    ]:
        data = load_json(run_dir / filename)
        if data is not None:
            flatten_json(prefix, data, row)
    return row

def add_decision_columns(df):
    out = df.copy()
    numeric_cols = [
        'evaluation.metrics.avg_ic',
        'evaluation.metrics.avg_ic_ci_lower',
        'evaluation.metrics.avg_spearman_corr',
        'evaluation.metrics.sharpe_top_20_newey_west',
        'evaluation.metrics.return_top_20',
        'evaluation.metrics.top_20_return_ci_lower',
        'evaluation.metrics.hit_rate',
        'evaluation.metrics.long_short_spread',
        'training.mean_best_val_ic',
        'training.models_trained',
    ]
    for col in numeric_cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors='coerce')
    score = pd.Series(0.0, index=out.index)
    weights = {
        'evaluation.metrics.avg_ic': 0.35,
        'evaluation.metrics.avg_spearman_corr': 0.25,
        'evaluation.metrics.sharpe_top_20_newey_west': 0.25,
        'evaluation.metrics.return_top_20': 0.15,
    }
    for col, weight in weights.items():
        if col in out.columns:
            vals = out[col].astype(float)
            denom = vals.std(skipna=True)
            if denom and np.isfinite(denom) and denom > 0:
                score += weight * ((vals - vals.mean(skipna=True)) / denom).fillna(0.0)
    out['decision_score'] = score
    out['status'] = np.where(out['returncode'].eq(0), 'OK', 'FAILED')
    if 'evaluation.metrics.avg_ic_ci_lower' in out.columns:
        out['ic_ci_pass'] = out['evaluation.metrics.avg_ic_ci_lower'].gt(0)
    if 'evaluation.metrics.top_20_return_ci_lower' in out.columns:
        out['top20_ci_pass'] = out['evaluation.metrics.top_20_return_ci_lower'].gt(0)
    return out.sort_values(['status', 'decision_score'], ascending=[True, False])

## 6. Run Full Ablation Matrix

In [ ]:
manifest_path = ABLATION_ROOT / 'full_feature_factorial_manifest.json'
manifest_path.write_text(json.dumps({
    'ablations': ABLATIONS,
    'base_overrides': BASE_OVERRIDES,
    'factor_counts': {
        'momentum': len(MOMENTUM_FACTORS),
        'graph': len(GRAPH_FACTORS),
        'regime': len(REGIME_FACTORS),
        'primary_factorial_runs': sum(a['family'] == 'factorial' for a in ABLATIONS),
        'diagnostic_controls': sum(a['family'] == 'diagnostic' for a in ABLATIONS),
    },
    'budget': {
        'num_models': NUM_MODELS,
        'num_epochs': NUM_EPOCHS,
        'early_stopping_patience': EARLY_STOPPING_PATIENCE,
        'bootstrap_resamples': BOOTSTRAP_RESAMPLES,
    },
}, indent=2))

statuses = []
rows = []

for ablation in ABLATIONS:
    status = run_ablation(ablation)
    statuses.append(status)
    rows.append(collect_ablation(ablation, status))
    interim_df = add_decision_columns(pd.DataFrame(rows))
    interim_df.to_csv(ABLATION_ROOT / 'ablation_decision_table_interim.csv', index=False)

raw_df = pd.DataFrame(rows)
raw_path = ABLATION_ROOT / 'ablation_results_raw.csv'
raw_df.to_csv(raw_path, index=False)

decision_df = add_decision_columns(raw_df)
decision_path = ABLATION_ROOT / 'ablation_decision_table.csv'
decision_df.to_csv(decision_path, index=False)

html_path = ABLATION_ROOT / 'ablation_decision_table.html'
decision_df.to_html(html_path, index=False)

print('Saved manifest:', manifest_path)
print('Saved raw results:', raw_path)
print('Saved decision table:', decision_path)
print('Saved HTML table:', html_path)

display_cols = [c for c in [
    'status',
    'name',
    'family',
    'decision_score',
    'evaluation.metrics.avg_ic',
    'evaluation.metrics.avg_ic_ci_lower',
    'evaluation.metrics.avg_spearman_corr',
    'evaluation.metrics.sharpe_top_20_newey_west',
    'evaluation.metrics.return_top_20',
    'evaluation.metrics.top_20_return_ci_lower',
    'training.mean_best_val_ic',
    'run_dir',
] if c in decision_df.columns]
display(decision_df[display_cols])

## 7. Main Effects and Interaction Tables

In [ ]:
METRICS_FOR_EFFECTS = [
    'decision_score',
    'evaluation.metrics.avg_ic',
    'evaluation.metrics.avg_ic_ci_lower',
    'evaluation.metrics.avg_spearman_corr',
    'evaluation.metrics.return_top_20',
    'evaluation.metrics.top_20_return_ci_lower',
    'evaluation.metrics.sharpe_top_20_newey_west',
]
METRICS_FOR_EFFECTS = [c for c in METRICS_FOR_EFFECTS if c in decision_df.columns]

ok_factorial = decision_df[(decision_df['status'].eq('OK')) & (decision_df['family'].eq('factorial'))].copy()

effect_tables = {}
for factor in ['momentum_factor', 'graph_factor', 'regime_factor']:
    if not ok_factorial.empty:
        effect = ok_factorial.groupby(factor)[METRICS_FOR_EFFECTS].agg(['mean', 'median', 'count'])
        effect_tables[factor] = effect
        path = ABLATION_ROOT / f'{factor}_main_effects.csv'
        effect.to_csv(path)
        print('Saved:', path)
        display(effect)

if not ok_factorial.empty and METRICS_FOR_EFFECTS:
    interaction_pairs = [
        ('momentum_factor', 'graph_factor'),
        ('momentum_factor', 'regime_factor'),
        ('graph_factor', 'regime_factor'),
    ]
    for left, right in interaction_pairs:
        pivot = ok_factorial.pivot_table(
            index=left,
            columns=right,
            values='decision_score',
            aggfunc='mean',
        )
        path = ABLATION_ROOT / f'interaction__{left}__{right}.csv'
        pivot.to_csv(path)
        print('Saved:', path)
        display(pivot)
else:
    print('No successful factorial runs available for effect tables yet.')

## 8. Visualize The Decision Table

In [ ]:
import matplotlib.pyplot as plt

plot_df = decision_df[decision_df['status'].eq('OK')].copy()
metric_cols = [
    'decision_score',
    'evaluation.metrics.avg_ic',
    'evaluation.metrics.avg_spearman_corr',
    'evaluation.metrics.sharpe_top_20_newey_west',
    'evaluation.metrics.return_top_20',
]
metric_cols = [c for c in metric_cols if c in plot_df.columns]

if not plot_df.empty and metric_cols:
    fig, axes = plt.subplots(len(metric_cols), 1, figsize=(16, 4 * len(metric_cols)))
    if len(metric_cols) == 1:
        axes = [axes]
    for ax, col in zip(axes, metric_cols):
        ordered = plot_df.sort_values(col, ascending=False)
        ax.bar(ordered['name'], ordered[col])
        ax.set_title(col)
        ax.tick_params(axis='x', rotation=75)
        ax.axhline(0, color='black', linewidth=0.8, alpha=0.4)
    plt.tight_layout()
    plot_path = ABLATION_ROOT / 'ablation_metric_bars.png'
    plt.savefig(plot_path, dpi=180, bbox_inches='tight')
    plt.show()
    print('Saved plot:', plot_path)
else:
    print('No successful runs or numeric metrics available to plot.')

## 9. Inspect Failed Runs

In [ ]:
failed = [s for s in statuses if s['returncode'] != 0]
print('Failed runs:', len(failed))
for status in failed:
    run_dir = Path(status['run_dir'])
    print('\n' + '=' * 100)
    print(status['name'])
    print('Run dir:', run_dir)
    stderr_path = run_dir / 'stderr.log'
    stdout_path = run_dir / 'stdout.log'
    if stderr_path.exists():
        print('stderr tail:')
        print(stderr_path.read_text()[-4000:])
    if stdout_path.exists():
        print('stdout tail:')
        print(stdout_path.read_text()[-4000:])

## 10. Export Summary Report

In [ ]:
report_path = ABLATION_ROOT / 'ablation_summary_report.md'
top_rows = decision_df.head(10).copy()
report_lines = [
    '# MCI-GRU Full Feature Factorial Ablation Summary',
    '',
    f'Run root: `{ABLATION_ROOT}`',
    f'Data file: `{DATA_FILE}`',
    f'Budget: {NUM_MODELS} models, {NUM_EPOCHS} epochs, early stopping patience {EARLY_STOPPING_PATIENCE}, {BOOTSTRAP_RESAMPLES} bootstrap resamples.',
    '',
    '## Top Runs',
    '',
    top_rows[[c for c in [
        'status',
        'name',
        'family',
        'decision_score',
        'evaluation.metrics.avg_ic',
        'evaluation.metrics.avg_ic_ci_lower',
        'evaluation.metrics.return_top_20',
        'evaluation.metrics.top_20_return_ci_lower',
        'run_dir',
    ] if c in top_rows.columns]].to_markdown(index=False),
    '',
    '## Output Files',
    '',
    f'- Manifest: `{manifest_path}`',
    f'- Raw results: `{raw_path}`',
    f'- Decision table: `{decision_path}`',
    f'- HTML table: `{html_path}`',
    f'- Metric bars: `{ABLATION_ROOT / "ablation_metric_bars.png"}`',
]
report_path.write_text('\n'.join(report_lines))
print('Saved report:', report_path)
print(report_path.read_text()[:4000])